In [53]:
import httpx
from httpx import Response
from IPython.display import JSON
import logging
import time

OMLX_API_URL = "http://127.0.0.1:8321/v1" # OpenAI API compatible
OMLX_API_KEY = "changeme"
OMLX_MODEL = "Qwen3.6-35B-A3B-4bit"

In [ ]:
# Calling the LLM
# official docs : 
# https://developers.openai.com/api/reference/overview
# https://docs.ollama.com/api/openai-compatibility
# https://developers.openai.com/api/docs/guides/function-calling?api-mode=chat


# Transform the function into the tool format expected by the LLM
# decorator to convert a function into a tool definition
def tool(func):
    return {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": func.__doc__,
            "parameters": {
                "type": "object",
                "properties": {
                    # self, cls are not included in the arguments list
                    # arg type can be inferred from type hints if available, otherwise default to string
                    arg: {"type": "string"} for arg in func.__code__.co_varnames[:func.__code__.co_argcount] if arg not in ('self', 'cls')
                },
                # ignore self, cls and any *args, **kwargs
                "required": [arg for arg in func.__code__.co_varnames[:func.__code__.co_argcount] if arg not in ('self', 'cls') and not arg.startswith('*')]
            }
        }
    }


class ToolsRegistry:
    def get_weather(self, location):
        """ Get the current weather for a location. """
        return f"The current weather in {location} is sunny with a temperature of 25 degrees Celsius."

class LLM:
    def __init__(self, api_url=OMLX_API_URL, api_key=OMLX_API_KEY, model=OMLX_MODEL):
        self.api_url = api_url
        self.api_key = api_key
        self.model = model
        self.setup_logger()
        self.setup_session()
        self.available_models()
    
    def setup_session(self):
        self.session = httpx.Client()
        self.session.headers.update({
            "content-type": "application/json",
            "Authorization": "Bearer " + self.api_key
        })
        self.session.timeout = 30.0
        self.logger.info("HTTP session initialized with API key and content type")
    
    def setup_logger(self):
        self.logger = logging.getLogger("LLM")
        for handler in self.logger.handlers[:]:
            self.logger.removeHandler(handler)
        self.logger.setLevel(logging.DEBUG)
        handler = logging.StreamHandler()
        handler.setLevel(logging.DEBUG)
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        handler.setFormatter(formatter)
        self.logger.addHandler(handler)
        self.logger.info("Logger initialized for LLM class")
    
    def available_models(self):
        resp = self.session.get(self.api_url+'/models')
        data = resp.json()
        models = [model['id'] for model in data.get('data', [])]
        self.logger.debug("Available models: %s", models)
        return models
    
    def chat_completion(self, messages, tools=[]):
        self.logger.info("Request sent to %s with model %s", self.api_url+'/chat/completions', self.model)
        resp = self.session.post(self.api_url+'/chat/completions', json={
            "model": self.model,
            "messages": messages,
            "tools": tools
        })
        self.logger.info("Received response: %s (%d bytes)", resp.status_code, len(resp.content))
        message = resp.json().get('choices', [{}])[0].get('message', {})
        return message

    def execute_tool_calls(self, message):
        tool_calls = message.get('tool_calls', [])
        results = []
        for tool_call in tool_calls or []:
            if tool_call['type'] == 'function':
                func_name = tool_call['function']['name']
                args = tool_call['function']['arguments']
                args = json.loads(args) if isinstance(args, str) else args
                self.logger.info("Processing tool call: %s with args %s ", func_name, args)
                ToolsClass = ToolsRegistry()
                if hasattr(ToolsClass, func_name):
                    func = getattr(ToolsClass, func_name)
                    result = func(**args) # execute the tool function with the provided arguments
                    results.append({
                        "role": "tool",
                        "tool_call_id": tool_call['id'],
                        "content": result
                    })
                    self.logger.info("Executed tool: %s with args %s", func_name, args)
                else:
                    self.logger.warning("Tool function %s not found in Tools class", func_name)
            else:
                self.logger.warning("Non-function tool calls are not supported in this implementation")
        return results
        
llm = LLM()

messages = [
    {
        "role": "system", 
        "content": "You are a helpful assistant. Answer the user's question based on the tools you have. Here are the Python tools you have access to: 1. get_weather(location) - Get the current weather for a given location."
    },
    {
        "role": "user", 
        "content": "Who is lebron james?"
    }
]
print("messages:", json.dumps(messages, indent=4))

# 0. Define the tools that the LLM can use based on the functions in the Tools class
tools = [tool(ToolsRegistry.get_weather)]
print("tools:", json.dumps(tools, indent=4))
time.sleep(1)

# 1. Ask the LLM a question, and tell to answer based on the tools it has access to.
# 2. LLM responds with a message that includes a tool call
message = llm.chat_completion(messages, tools=tools)
print("message:", json.dumps(message, indent=4))
messages.append(message)
time.sleep(1)

# 3. Execute any tool calls that the LLM requested for the message
results = llm.execute_tool_calls(message)
print("results:", json.dumps(results, indent=4))
# 4. Add the tool results back to the messages list as tool responses
messages.extend(results)
print("messages:", json.dumps(messages, indent=4))
time.sleep(1)

# 5. Send the updated messages back to the LLM to get a final response that includes the tool results
final_message = llm.chat_completion(messages)
print("final_message:", json.dumps(final_message, indent=4))



2026-05-22 01:28:00,913 - LLM - INFO - Logger initialized for LLM class
2026-05-22 01:28:00,918 - LLM - INFO - HTTP session initialized with API key and content type
2026-05-22 01:28:00,921 - LLM - DEBUG - Available models: ['MLX-Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled-v2-4bit', 'MLX-Qwen3.5-9B-Claude-4.6-Opus-Reasoning-Distilled-v2-4bit', 'Qwen3.5-27B-Claude-4.6-Opus-Distilled-MLX-4bit', 'Qwen3.5-9B-MLX-4bit', 'Qwen3.6-27B-4bit', 'Qwen3.6-35B-A3B-4bit', 'Qwen3.6-35B-A3B-5bit', 'Qwen3.6-35B-A3B-6bit', 'gemma-4-26b-a4b-it-4bit', 'gemma-4-31b-it-4bit', 'gemma-4-e2b-it-4bit', 'gpt-oss-20b-MXFP4-Q4', 'gpt-oss-20b-MXFP4-Q8']


messages: [
    {
        "role": "system",
        "content": "You are a helpful assistant. Answer the user's question based on the tools you have. Here are the Python tools you have access to: 1. get_weather(location) - Get the current weather for a given location."
    },
    {
        "role": "user",
        "content": "Who is lebron james?"
    }
]
tools: [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": " Get the current weather for a location. ",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string"
                    }
                },
                "required": [
                    "location"
                ]
            }
        }
    }
]


2026-05-22 01:28:01,923 - LLM - INFO - Request sent to http://127.0.0.1:8321/v1/chat/completions with model Qwen3.6-35B-A3B-4bit
2026-05-22 01:28:08,102 - LLM - INFO - Received response: 200 (2398 bytes)


message: {
    "role": "assistant",
    "content": "LeBron James is an American professional basketball player widely regarded as one of the greatest players in NBA history. Born in 1984, he has played for the Cleveland Cavaliers, Miami Heat, and Los Angeles Lakers throughout his career. \n\nHis major accomplishments include:\n* **4\u00d7 NBA Champion** (2012, 2013 with Miami; 2016 with Cleveland; 2020 with LA)\n* **4\u00d7 NBA Most Valuable Player** (2012, 2013, 2014, 2016)\n* **2\u00d7 Olympic Gold Medalist** (2008, 2012)\n* **All-time leading scorer** in NBA history (regular season and playoffs combined)\n* Multiple All-Star selections, scoring titles, and Finals MVP awards\n\nOff the court, he is also known for his philanthropy, particularly through his \"I PROMISE\" school in Akron, Ohio, and his media and business ventures.",
    "reasoning_content": "Thinking Process:\n1.  **Analyze User Input:** The user asks \"Who is lebron james?\"\n2.  **Identify Intent:** The user is asking

2026-05-22 01:28:10,109 - LLM - INFO - Request sent to http://127.0.0.1:8321/v1/chat/completions with model Qwen3.6-35B-A3B-4bit
2026-05-22 01:28:15,313 - LLM - INFO - Received response: 200 (1968 bytes)


final_message: {
    "role": "assistant",
    "content": "LeBron James is an American professional basketball player widely regarded as one of the greatest players in NBA history. Born in 1984, he has played for the Cleveland Cavaliers, Miami Heat, and Los Angeles Lakers throughout his career. \n\nHis major accomplishments include:\n* **4\u00d7 NBA Champion** (2012, 2013 with Miami; 2016 with Cleveland; 2020 with LA)\n* **4\u00d7 NBA Most Valuable Player** (2012, 2013, 2014, 2016)\n* **2\u00d7 Olympic Gold Medalist** (2008, 2012)\n* **All-time leading scorer** in NBA history (regular season and playoffs combined)\n* Multiple All-Star selections, scoring titles, and Finals MVP awards\n\nOff the court, he is also known for his philanthropy, particularly through his \"I PROMISE\" school in Akron, Ohio, and his media and business ventures.",
    "reasoning_content": "The user is asking a general knowledge question about LeBron James.\nI have a tool `get_weather(location)` but it is irrelev